# RAG Minecraft Wiki — Pipeline completo en un notebook

Este notebook replica todo el pipeline RAG del proyecto, celda por celda, ejecutable de arriba a abajo. Sirve para **entender el flujo** sin saltar entre archivos.

El sistema responde preguntas sobre Minecraft usando la [Minecraft Wiki oficial](https://minecraft.wiki) (licencia CC BY-SA).

1. **Ingesta** (offline): wiki → texto → chunks → embeddings → ChromaDB
2. **Query** (runtime): pregunta → embedding → búsqueda semántica → contexto → Claude → respuesta

> ⚠️ Este notebook comparte la base de datos (`data/chroma/`) con la API. Si ejecutas la celda de reset, borras lo que la API tiene.

## 0. Setup: imports y configuración

In [ ]:
# 📚 truststore usa los certificados nativos de Windows.
# Sin esto, huggingface_hub falla con SSLCertVerificationError.
import truststore
truststore.inject_into_ssl()

import os
import sys
import time
from pathlib import Path
from typing import NamedTuple

import mwclient
import mwparserfromhell
import chromadb
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

# --- Configuración ---
BASE_DIR = Path.cwd().parent
WIKI_PATH = BASE_DIR / "data" / "wiki"
CHROMA_PATH = BASE_DIR / "data" / "chroma"
COLLECTION_NAME = "minecraft"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

WIKI_USER_AGENT = "RAG-Minecraft-Notebook/1.0 (contact: argomezflores@gmail.com)"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 5

WIKI_PATH.mkdir(parents=True, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"API key configurada: {'sí' if ANTHROPIC_API_KEY else '❌ NO (edita .env)'}")

## 1. Scraping de la Minecraft Wiki

**Concepto**: usamos la API MediaWiki (`https://minecraft.wiki/api.php`) que devuelve wikitext plano. La librería `mwclient` la envuelve; con `mwparserfromhell` limpiamos `{{templates}}` y `[[links]]` a texto plano.

**Ética**: User-Agent identificable + 1s delay entre requests.

> ⚠️ Este ejemplo baja **solo 5 páginas** para no tardar. Para full corpus, corre `python scripts/scrape_wiki.py` fuera del notebook.

In [ ]:
site = mwclient.Site("minecraft.wiki", path="/", clients_useragent=WIKI_USER_AGENT)
print(f"Conectado a {site.host} (versión {site.version})")


def clean_wikitext(raw: str) -> str:
    wikicode = mwparserfromhell.parse(raw)
    text = wikicode.strip_code(normalize=True, collapse=True)
    return text.strip()


# Bajamos 5 páginas de ejemplo
SAMPLE_PAGES = ["Beacon", "Diamond Pickaxe", "Ender Dragon", "Nether portal", "Enchanting"]
sample_articles = []
for title in SAMPLE_PAGES:
    page = site.pages[title]
    raw = page.text()
    clean = clean_wikitext(raw)
    sample_articles.append({
        "title": title,
        "url": f"https://minecraft.wiki/w/{title.replace(' ', '_')}",
        "text": clean,
    })
    print(f"[ok] {title} — {len(clean)} chars limpios")
    time.sleep(1.0)

print(f"\n{len(sample_articles)} artículos en memoria")
print(f"\nPrimeros 300 chars de 'Beacon':\n{sample_articles[0]['text'][:300]}")

## 2. Chunker: dividir texto en fragmentos manejables

**Concepto RAG**: los embeddings tienen límite de ~512 tokens ≈ 2000 chars. Dividimos en chunks de 800 chars con overlap de 150 para no cortar ideas a la mitad.

In [ ]:
def split_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []
    separators = ["\n\n", "\n", ". ", " "]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end >= len(text):
            chunks.append(text[start:].strip())
            break
        split_at = -1
        for sep in separators:
            idx = text.rfind(sep, start, end)
            if idx > start:
                split_at = idx + len(sep)
                break
        if split_at == -1:
            split_at = end
        chunks.append(text[start:split_at].strip())
        start = max(split_at - overlap, start + 1)
    return [c for c in chunks if c]

# Test: dividimos el artículo más largo
longest = max(sample_articles, key=lambda a: len(a['text']))
chunks = split_text(longest['text'])
print(f"Artículo: {longest['title']} ({len(longest['text'])} chars) → {len(chunks)} chunks")
print(f"\nPrimer chunk (200 chars):\n{chunks[0][:200]}...")

## 3. Embedder: texto → vector de 384 dimensiones

Usamos `all-MiniLM-L6-v2` — inglés puro (mejor calidad que multilingüe para EN), local, gratis.

In [ ]:
print(f"Cargando modelo {EMBEDDING_MODEL}...")
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Listo. Dimensión: {model.get_sentence_embedding_dimension()}")


def embed_texts(texts: list[str], batch_size: int = 32) -> list[list[float]]:
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=len(texts) > 50,
        normalize_embeddings=True,
    )
    return vectors.tolist()


# Test: 3 frases sobre Minecraft
import numpy as np
vecs = embed_texts([
    "How do I craft a diamond pickaxe?",
    "What materials are needed for a diamond pickaxe?",
    "The weather forecast is sunny today.",
])
v = np.array(vecs)
print(f"\nSimilitud coseno (0-1):")
print(f"  1 vs 2 (mismo tema): {np.dot(v[0], v[1]):.3f}")
print(f"  1 vs 3 (no relacionado): {np.dot(v[0], v[2]):.3f}")

## 4. Ingesta a ChromaDB

Guardamos: texto + vector + metadata (`title`, `url`, `category`).

In [ ]:
client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# 📚 Reset para este ejemplo. Ojo: si tu API ya tiene datos, comenta esto.
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Colección '{COLLECTION_NAME}' eliminada.")
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

total = 0
for article in sample_articles:
    chunks = split_text(article['text'], CHUNK_SIZE, CHUNK_OVERLAP)
    if not chunks:
        continue
    safe_title = article['title'].replace(' ', '_').replace('/', '_')
    ids = [f"{safe_title}_c{i}" for i in range(len(chunks))]
    metas = [
        {"title": article['title'], "url": article['url'], "category": "Sample", "chunk_index": i}
        for i in range(len(chunks))
    ]
    embeddings = embed_texts(chunks)
    collection.upsert(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metas)
    total += len(chunks)
    print(f"  {article['title']}: {len(chunks)} chunks")

print(f"\n✅ Total: {total} chunks en '{COLLECTION_NAME}'")
print(f"Chunks en Chroma: {collection.count()}")

## 5. Retrieval: query → top-K chunks

1. La query se convierte al mismo espacio vectorial.
2. Chroma busca los K más cercanos.
3. Devolvemos texto + metadata + score.

In [ ]:
class Hit(NamedTuple):
    text: str
    title: str
    url: str
    category: str
    score: float


def search(query: str, k: int = TOP_K) -> list[Hit]:
    q_vec = embed_texts([query])[0]
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        hits.append(Hit(
            text=doc,
            title=meta.get('title', '?'),
            url=meta.get('url', ''),
            category=meta.get('category', ''),
            score=round(1.0 - (dist / 2.0), 3),
        ))
    return hits


# Test
question = "How do I craft a Beacon?"
hits = search(question, k=3)
print(f"Q: {question}\n")
for i, h in enumerate(hits, 1):
    print(f"[{i}] {h.title} · score={h.score}")
    print(f"    URL: {h.url}")
    print(f"    {h.text[:200].strip()}...\n")

## 6. Generación con Claude (respuestas en español)

System prompt clave: "responde en ESPAÑOL aunque el contexto esté en inglés, sé conciso (chat game), cita fuentes con [N]".

In [ ]:
SYSTEM_PROMPT = """Eres un experto en Minecraft (Java Edition, versiones modernas 1.20+).
Respondes preguntas de jugadores usando ÚNICAMENTE el contexto oficial de la Minecraft Wiki.

Reglas:
1. Responde en ESPAÑOL aunque el contexto esté en inglés.
2. Máx 4-5 líneas (el chat del juego es corto).
3. Menciona items/mobs por su nombre en inglés ("Diamond Pickaxe", "Enderman").
4. Si no está en el contexto, di: "No tengo información sobre eso en la wiki."
5. Números y stats siempre precisos.
6. Cita fuentes con [N]."""


def generate_answer(question: str, hits: list[Hit]) -> str:
    if not ANTHROPIC_API_KEY:
        return "❌ ANTHROPIC_API_KEY no configurada."
    if not hits:
        return "No tengo información sobre eso en la wiki."

    context = "\n\n---\n\n".join(
        f"[{i}] (Wiki: {h.title})\n{h.text}"
        for i, h in enumerate(hits, start=1)
    )
    user_msg = f"Contexto:\n{context}\n\n---\n\nPregunta: {question}\n\nResponde en español, cita fuentes con [N]."

    client_anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
    response = client_anthropic.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )
    return response.content[0].text

## 7. Pipeline completo end-to-end 🚀

In [ ]:
def rag_query(question: str, k: int = TOP_K) -> dict:
    hits = search(question, k=k)
    answer = generate_answer(question, hits)
    return {
        'question': question,
        'answer': answer,
        'sources': [
            {'title': h.title, 'url': h.url, 'category': h.category, 'score': h.score}
            for h in hits
        ],
    }


result = rag_query("¿Cómo se craftea un Beacon?")

print("PREGUNTA:", result['question'])
print("\nRESPUESTA:\n", result['answer'])
print("\nFUENTES:")
for i, s in enumerate(result['sources'], 1):
    print(f"  [{i}] {s['title']} — {s['url']} (score={s['score']})")

## 8. Experimenta 🧪

Prueba tus propias preguntas. Como este notebook solo tiene 5 artículos de ejemplo, para preguntas fuera de esas usa el corpus completo (`python scripts/scrape_wiki.py` + `python scripts/ingest_all.py --reset`).

In [ ]:
my_question = "¿Qué necesito para derrotar al Ender Dragon?"
result = rag_query(my_question, k=5)

print("PREGUNTA:", result['question'])
print("\nRESPUESTA:\n", result['answer'])
print("\nFUENTES:")
for i, s in enumerate(result['sources'], 1):
    print(f"  [{i}] {s['title']} (score={s['score']})")

## 9. 🆕 Bonus v3: Query Rewriting con Claude Haiku

Antes del retrieval, un LLM rápido (Haiku) genera 3 variaciones de la pregunta. Cada una captura un ángulo distinto. Después buscamos por las 3 y hacemos merge.

**Impacto**: preguntas en español sobre corpus en inglés mejoran su cobertura. En vez de 1 embedding para buscar, tienes 3 ángulos.

In [ ]:
import json
import re

HAIKU_MODEL = 'claude-haiku-4-5-20251001'

REWRITE_PROMPT = '''Eres un asistente que reformula preguntas de jugadores de Minecraft para mejorar la búsqueda en la wiki oficial (en inglés).

Genera 3 REFORMULACIONES: una directa, una técnica en inglés, una específica.
Devuelve SOLO un JSON array con 3 strings. Sin markdown, sin explicación.'''


def rewrite_query(question: str) -> list[str]:
    if not ANTHROPIC_API_KEY:
        return [question]
    try:
        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=HAIKU_MODEL, max_tokens=300,
            system=REWRITE_PROMPT,
            messages=[{'role': 'user', 'content': question}],
        )
        text = resp.content[0].text
        match = re.search(r'\[[\s\S]*?\]', text)
        rewrites = json.loads(match.group(0)) if match else []
        return [question] + [r for r in rewrites if r != question][:3]
    except Exception as e:
        print(f'fallback: {e}')
        return [question]


# Demo
queries = rewrite_query('¿cómo funciona el repeater?')
for i, q in enumerate(queries, 1):
    marker = '📌 Original' if i == 1 else f'✨ Rewrite #{i-1}'
    print(f'{marker}: {q}')

## 10. 🆕 Bonus v3: Re-ranking con Cross-Encoder

Después del retrieval bi-encoder (rápido, imperciso), un cross-encoder re-puntúa los top-N candidatos viendo (query, chunk) como PAR. Mucho más preciso.

**El problema que resuelve**: la query "Diamond Pickaxe" traía chunks de Anvil (porque menciona diamond). Con re-ranking, salen los chunks del artículo Diamond Pickaxe mismo.

> ⚠️ El cross-encoder se descarga la primera vez (~90 MB).

In [ ]:
from sentence_transformers import CrossEncoder
from math import exp

print('Cargando cross-encoder (~90 MB primera vez)...')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Listo.')


def sigmoid(x): return 1.0 / (1.0 + exp(-x))


def rerank(query, hits, top_k=5):
    if not hits: return []
    pairs = [(query, h.text) for h in hits]
    scores = reranker.predict(pairs)
    scored = sorted(zip(hits, scores), key=lambda p: p[1], reverse=True)[:top_k]
    return [Hit(text=h.text, title=h.title, url=h.url, category=h.category,
                score=round(sigmoid(float(s)), 3)) for h, s in scored]


# Demo: comparación lado a lado
test_query = 'How do I craft a Beacon?'
before = search(test_query, k=5)
after = rerank(test_query, before, top_k=5)

print('\n📊 SIN re-ranking (bi-encoder):')
for i, h in enumerate(before, 1):
    print(f'  {i}. {h.title} — score={h.score}')

print('\n📊 CON re-ranking (cross-encoder):')
for i, h in enumerate(after, 1):
    print(f'  {i}. {h.title} — score={h.score}')